# Databricks Pyspark MERGE (Incremental Load)

**Incremental Load:-**
	Only new or updated records load into target table from source table.

**Source_table:**

ID 	Name 	Email_ID

1 	Bob	bob_new@example.com

3 	Carol	carol@example.com	

**target_table:**

ID 	Name 	Email_ID

1   	Alice	alice@example.com

2	   Bob	bob@example.com

**After Incremental Load**

ID	Name	Email_ID

1	Bob	bob_new@example.com

2	Bob	bob@example.com

3	Carol	carol@example.com











In [0]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

In [0]:
spark = SparkSession.builder.appName("MergeExample").getOrCreate()

In [0]:
#Target Data

target_data = [(1,"Alice","alice@example.com"),
	     (2,"Bob","bob@example.com")
	    ]

#Source Data
source_data =[(1,"Bob","bob_new@example.com"),#update
	     (3,"Carol","carol@example.com") #insert
	   ]

columns = ["id","name","email"]

target_df = spark.createDataFrame(target_data,columns)
source_df=spark.createDataFrame(source_data,columns)

#Convert to Delta Table (Required for MERGE)
target_df.write.format("delta").mode("overwrite").saveAsTable("training.default.target_table")


In [0]:
target_delta = DeltaTable.forName(spark,"training.default.target_table")

#MERGE(UPSERT)
target_delta.alias("t") \
    .merge(source_df.alias("s"),"t.id=s.id") \
        .whenMatchedUpdate(set={"name":"s.name",
                                "email":"s.email"}) \
        .whenNotMatchedInsert(values={"id":"s.id",
                                   "name":"s.name",
                                   "email":"s.email"}) \
        .execute()